In [1]:
import sys
import os
import importlib

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

# Force-reload modules so edits are picked up without kernel restart
import llm, agents.specialist, agents.orchestrator, agents.verifier, state
importlib.reload(llm)
importlib.reload(state)
importlib.reload(agents.specialist)
importlib.reload(agents.orchestrator)
importlib.reload(agents.verifier)

# Core imports
from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import (
    search_flight_candidates, search_hotel_candidates, search_activity_candidates,
    search_and_save_candidates, select_top_k, 
    # Legacy
    run_activities_specialist, run_logistics_specialist,
)
from agents.verifier import (
    verify_and_format_itinerary, format_complete_itinerary,
    validate_plan, format_negotiation_message,
)
from state import TravelState, TravelNeeds

# Benchmark imports
sys.path.insert(0, os.path.join(notebook_dir, "..", "benchmark", "src"))
import run_benchmark
importlib.reload(run_benchmark)
from run_benchmark import (
    load_orchestrator_tasks, filter_tasks, run_single_task,
    run_benchmark, run_evaluation, run_multi_turn_task,
)

# Example: Run 1 Task → Evaluate → Restore Plan
Run a single benchmark task through the full pipeline, evaluate the result, and display the saved plan.

In [6]:
# =============================================
# Run 1 task → Evaluate → Restore Plan
# =============================================
# Change TASK_ID to any task: ORCH-T1-01..06, ORCH-T2-01..08, ORCH-T3-01..06
TASK_ID = "ORCH-T1-01"

# --- 1. Run the task through the agent pipeline ---
tasks = load_orchestrator_tasks()
task_list = filter_tasks(tasks, task_id=TASK_ID)

if not task_list:
    raise ValueError(f"Task {TASK_ID} not found!")

result = run_single_task(task_list[0], verbose=True)
print(f"\n{'='*50}")
print(f"Status: {result['status']}  |  Time: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"Error: {result['error']}")

# --- 2. Evaluate the saved plan ---
from tools.auto_evaluator import AutoEvaluator

evaluator = AutoEvaluator()
eval_result = evaluator.evaluate_single_plan(TASK_ID)

print(f"\n{'='*50}")
print(f"EVALUATION for {TASK_ID}")
print(f"  Overall Score:    {eval_result.get('overall_score', 0):.1%}")
print(f"  Schema Compliance:{eval_result.get('schema_compliance', 0):.1%}")
print(f"  CSR (fractional): {eval_result.get('csr_score', 0):.1%}")
csr = eval_result.get('csr_score', 0)
print(f"  CSR (binary):     {'PASS' if csr >= 1.0 else 'FAIL'}")
print(f"  Tool Accuracy:    {eval_result.get('tool_accuracy', 0):.1%}")
int_s = eval_result.get('interest_score')
if int_s is not None:
    print(f"  Interest Score:   {int_s:.1%}")
    breakdown = eval_result.get('interest_breakdown', {})
    for k, v in (breakdown or {}).items():
        print(f"    {'✓' if v else '✗'} {k}")

# --- 3. Restore & display the saved plan ---
from tools.plan_repository import PlanRepository
from tools.plan_schema import format_plan_summary

repo = PlanRepository()
saved_plan = repo.load_plan_with_metadata(TASK_ID)

print(format_plan_summary(saved_plan))


[ORCH-T1-01] (Tier 1) Flight with preference and morning time window
  Query: Find a round-trip flight from Boston to Seattle, departing May 5 and returning May 12, 2026. I stron...
  [LLM CACHE HIT] 76c068aa6ab21985
  Orchestrator → intent=new_trip, needs: flight=True, hotel=False, activity=False
  Searching flights...
  [Search] Flights: Boston(BOS) -> Seattle(SEA) (2026-05-05 to 2026-05-12)
  [Cache Hit] google_flights
  [Saved] flights candidates -> c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\search_candidates\ORCH-T1-01\flights_candidates.json (2 total)
  [Found] 19 flight options
  [flights] 1 search call(s)
  Running top-K selection...
  [Selector] LLM selecting from 5 flights, 0 hotels, 0 activities...
  [LLM CACHE HIT] ee5c18bb393e36f6
  Validating plan...
✓ Plan saved: c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\plans\ORCH-T1-01\plan.json
  ✓ SAVED (valid) → c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\plans\ORCH-T1-01\plan.json
  ✓ Needs matc

## Benchmark Runner
Run benchmark tasks from `orchestrator_tasks.json` through the full agent pipeline (Orchestrator → Specialist → Verifier → Save → Evaluate).

In [3]:
# =============================================
# Run ALL Tier-1 benchmark tasks
# =============================================
# Change tier=2 or tier=1 for other tiers
# Remove the tier filter to run ALL tasks

tasks = load_orchestrator_tasks()
tier = 3
select_tasks = filter_tasks(tasks, tier=tier)

print(f"Running {len(select_tasks)} Tier-{tier} tasks...\n")
results = run_benchmark(select_tasks, verbose=True)



Running 18 Tier-3 tasks...


############################################################
  BENCHMARK: Running 18 tasks (6 conversations)
############################################################

############################################################
  MULTI-TURN: ORCH-T3-01 (3 turns)
############################################################

[ORCH-T3-01-turn1] (Tier 3) Progressive NYC trip: base plan (turn 1 of 3)
  Query: Plan a 3-day trip to NYC from Boston under $1,200, departing June 1, 2026. Just flights and a hotel ...
  [LLM CACHE SAVE] 6dcb303dd1a19825
  Orchestrator → intent=new_trip, needs: flight=True, hotel=True, activity=False
  Searching flights...
  [Search] Flights: Boston(BOS) -> New York(JFK) (2026-06-01 to 2026-06-04)
  [API Call] google_flights...
  [Saved] google_flights -> c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\cache
  [Saved] flights candidates -> c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\search_candidates\ORCH-T3-01-tur

In [4]:
# =============================================
# Evaluate all saved plans
# =============================================
# Run this after 7a or 7b to score the generated plans

eval_results = run_evaluation(verbose=True)


############################################################
  EVALUATION: Scoring all saved plans
############################################################

Found 31 plans to evaluate: ['ORCH-T1-01', 'ORCH-T1-02', 'ORCH-T1-03', 'ORCH-T1-04', 'ORCH-T1-05', 'ORCH-T1-06', 'ORCH-T2-01', 'ORCH-T2-02', 'ORCH-T2-03', 'ORCH-T2-04', 'ORCH-T2-05', 'ORCH-T2-06', 'ORCH-T2-07', 'ORCH-T2-08', 'ORCH-T3-01-turn1', 'ORCH-T3-01-turn2', 'ORCH-T3-01-turn3', 'ORCH-T3-02-turn1', 'ORCH-T3-02-turn2', 'ORCH-T3-02-turn3', 'ORCH-T3-03-turn1', 'ORCH-T3-03-turn2', 'ORCH-T3-04-turn1', 'ORCH-T3-04-turn2', 'ORCH-T3-04-turn3', 'ORCH-T3-05-turn1', 'ORCH-T3-05-turn2', 'ORCH-T3-05-turn3', 'ORCH-T3-05-turn4', 'ORCH-T3-06-turn1', 'ORCH-T3-06-turn2']


[AutoEvaluator] Evaluating 31 plans...
[AutoEvaluator] Evaluating ORCH-T1-01... ✓ Loaded plan from repository: ORCH-T1-01
✓ Score: 100.0%
[AutoEvaluator] Evaluating ORCH-T1-02... ✓ Loaded plan from repository: ORCH-T1-02
✓ Score: 100.0%
[AutoEvaluator] Evaluating ORCH-T1

## Bootstrap Statistical Analysis
Compute bootstrap 95% CI, fractional/binary CSR, per-tier breakdown, and Cohen's d effect sizes from evaluation results.
Requires running cells 7a/7b first (to generate plans), then 7c (to score them).

In [5]:
# =============================================
# 7d. Bootstrap CI + Effect Sizes
# =============================================
# Uses eval_results from cell 7c (run that first)

import importlib, sys, os

# Import statistics module
benchmark_src = os.path.join(os.getcwd(), "..", "benchmark", "src")
if benchmark_src not in sys.path:
    sys.path.insert(0, benchmark_src)

import statistics as bench_stats
importlib.reload(bench_stats)
from statistics import bootstrap_ci, cohens_d

# --- Collect per-task metrics ---
overall_scores = []
csr_scores = []
tool_scores = []
interest_scores = []
binary_passes = []
tier_scores = {}  # tier label → [scores]

for task_id, r in sorted(eval_results.items()):
    if "overall_score" not in r:
        continue
    os_val = r["overall_score"]
    csr_val = r.get("csr_score", 0)
    tool_val = r.get("tool_accuracy", 0)
    int_val = r.get("interest_score")

    overall_scores.append(os_val)
    csr_scores.append(csr_val)
    tool_scores.append(tool_val)
    if int_val is not None:
        interest_scores.append(int_val)
    binary_passes.append(1 if csr_val >= 1.0 else 0)

    parts = task_id.split("-")
    tier_label = parts[1] if len(parts) >= 2 else "unknown"
    tier_scores.setdefault(tier_label, []).append(os_val)

n = len(overall_scores)

# --- Bootstrap 95% CI ---
ci_overall = bootstrap_ci(overall_scores)
ci_csr     = bootstrap_ci(csr_scores)
ci_tools   = bootstrap_ci(tool_scores)
binary_rate = sum(binary_passes) / n if n else 0

print(f"{'='*60}")
print(f"BOOTSTRAP ANALYSIS  (n={n}, 1000 resamples, 95% CI)")
print(f"{'='*60}")
print(f"  Overall Score:  {ci_overall['mean']:.1%}  [{ci_overall['ci_lower']:.1%}, {ci_overall['ci_upper']:.1%}]  SD={ci_overall['std']:.3f}")
print(f"  CSR (frac):     {ci_csr['mean']:.1%}  [{ci_csr['ci_lower']:.1%}, {ci_csr['ci_upper']:.1%}]  SD={ci_csr['std']:.3f}")
print(f"  CSR (binary):   {binary_rate:.1%}  ({sum(binary_passes)}/{n} tasks fully satisfied)")
print(f"  Tool Accuracy:  {ci_tools['mean']:.1%}  [{ci_tools['ci_lower']:.1%}, {ci_tools['ci_upper']:.1%}]  SD={ci_tools['std']:.3f}")
if interest_scores:
    ci_int = bootstrap_ci(interest_scores)
    print(f"  Interest Score: {ci_int['mean']:.1%}  [{ci_int['ci_lower']:.1%}, {ci_int['ci_upper']:.1%}]  SD={ci_int['std']:.3f}  (n={len(interest_scores)} tasks)")

# --- Per-tier breakdown ---
tier_labels = sorted(tier_scores.keys())
if len(tier_labels) > 1:
    print(f"\n  Per-Tier Overall Score:")
    for tl in tier_labels:
        ci_t = bootstrap_ci(tier_scores[tl])
        print(f"    {tl} (n={len(tier_scores[tl])}):  {ci_t['mean']:.1%}  [{ci_t['ci_lower']:.1%}, {ci_t['ci_upper']:.1%}]")

# --- Effect sizes ---
effect_sizes = {}
for i in range(len(tier_labels)):
    for j in range(i + 1, len(tier_labels)):
        key = f"{tier_labels[i]} vs {tier_labels[j]}"
        d = cohens_d(tier_scores[tier_labels[i]], tier_scores[tier_labels[j]])
        effect_sizes[key] = d

if effect_sizes:
    print(f"\n  Effect Sizes (Cohen's d):")
    for key, d in effect_sizes.items():
        mag = "small" if abs(d) < 0.5 else "medium" if abs(d) < 0.8 else "large"
        print(f"    {key}: d={d:.3f} ({mag})")

# --- Per-task detail ---
print(f"\n{'─'*60}")
print(f"  Per-Task Breakdown:")
for task_id, r in sorted(eval_results.items()):
    if "overall_score" not in r:
        continue
    csr = r.get("csr_score", 0)
    tag = "PASS" if csr >= 1.0 else "FAIL"
    int_s = r.get("interest_score")
    int_str = f"  Int={int_s:.1%}" if int_s is not None else ""
    print(f"    {task_id:16}  Overall={r['overall_score']:.1%}  CSR={csr:.1%} [{tag}]  Tools={r.get('tool_accuracy',0):.1%}{int_str}")

print(f"{'='*60}")

BOOTSTRAP ANALYSIS  (n=31, 1000 resamples, 95% CI)
  Overall Score:  95.9%  [93.2%, 97.7%]  SD=0.064
  CSR (frac):     95.2%  [87.1%, 100.0%]  SD=0.195
  CSR (binary):   93.5%  (29/31 tasks fully satisfied)
  Tool Accuracy:  97.0%  [94.0%, 99.4%]  SD=0.081
  Interest Score: 89.6%  [81.8%, 97.0%]  SD=0.197  (n=25 tasks)

  Per-Tier Overall Score:
    T1 (n=6):  91.0%  [81.8%, 97.5%]
    T2 (n=8):  96.3%  [92.7%, 99.1%]
    T3 (n=17):  97.4%  [95.7%, 98.8%]

  Effect Sizes (Cohen's d):
    T1 vs T2: d=-0.581 (medium)
    T1 vs T3: d=-0.744 (medium)
    T2 vs T3: d=-0.241 (small)

────────────────────────────────────────────────────────────
  Per-Task Breakdown:
    ORCH-T1-01        Overall=100.0%  CSR=100.0% [PASS]  Tools=100.0%  Int=100.0%
    ORCH-T1-02        Overall=100.0%  CSR=100.0% [PASS]  Tools=100.0%  Int=100.0%
    ORCH-T1-03        Overall=68.5%  CSR=0.0% [FAIL]  Tools=100.0%  Int=100.0%
    ORCH-T1-04        Overall=92.5%  CSR=100.0% [PASS]  Tools=100.0%  Int=100.0%
    ORCH

## New Pipeline: Search → Select → Validate → Negotiate
The new pipeline replaces the old ReAct-based flow:
1. **Orchestrator**: Parse query → detect `needs` (flight/hotel/activity), default=false
2. **Search & Save**: Direct SerpAPI calls per category → save ALL raw data
3. **Top-K Selection**: Programmatic filter → LLM picks best combination
4. **Validate & Negotiate**: Check hard constraints/budget → if unmet, ask user

In [ ]:
# =============================================
# Step 1: Orchestrator — Parse query & detect needs
# =============================================
# Change the query to test different scenarios
query = "Find the cheapest round-trip flight from Boston to Seattle, departing May 5 and returning May 12, 2026."

state = TravelState()
analysis = analyze_user_input(query, state)
state = update_state_from_analysis(state, analysis)

print(f"Intent:      {analysis.get('intent')}")
print(f"Needs:       flight={state.needs.flight}, hotel={state.needs.hotel}, activity={state.needs.activity}")
print(f"Constraints: {state.constraints.model_dump_json(indent=2)}")
print(f"Response:    {analysis.get('response_to_user', '')[:200]}")

In [ ]:
# =============================================
# Step 2: Search & Save — Flight / Hotel / Activity independently
# =============================================
# Each search runs independently based on its own readiness check.
# Raw data saved to output/search_candidates/{task_id}/

search_results = {"flights": [], "hotels": [], "activities": []}

# --- Flight ---
if state.needs.flight:
    print(f"[Flight] ready={state.constraints.is_ready_for_flight()}")
    search_results["flights"] = search_flight_candidates(
        constraints=state.constraints, task_id=state.task_id,
    )

# --- Hotel ---
if state.needs.hotel:
    print(f"[Hotel]  ready={state.constraints.is_ready_for_hotel()}")
    search_results["hotels"] = search_hotel_candidates(
        constraints=state.constraints, task_id=state.task_id,
    )

# --- Activity ---
if state.needs.activity:
    print(f"[Activity] destination={'set' if state.constraints.destination else 'missing'}")
    search_results["activities"] = search_activity_candidates(
        constraints=state.constraints, task_id=state.task_id,
    )

# Summary
for cat, results in search_results.items():
    if results:
        print(f"\n[{cat.upper()}] {len(results)} search call(s) completed")

In [ ]:
# =============================================
# Step 3: Top-K Selection with LLM
# =============================================
# Programmatic filter to top-K per category, then LLM picks best combo

constraints_str = state.constraints.model_dump_json(indent=2)

selection = select_top_k(
    task_id=state.task_id,
    constraints_json=constraints_str,
    needs=state.needs,
    search_results=search_results,
    top_k=5,
)

print(f"\nConstraints met: {selection.get('constraints_met')}")
print(f"Unmet: {selection.get('unmet_constraints', [])}")
print(f"Reasoning: {selection.get('reasoning', '')[:300]}")
print(f"\nEstimated cost: ${selection.get('itinerary', {}).get('estimated_total_cost', 'N/A')}")

In [ ]:
# =============================================
# Step 4: Validate & Negotiate
# =============================================
# Check hard constraints and budget. If unmet, shows negotiation options.

validation = validate_plan(selection, state.constraints)

if validation["valid"]:
    print("✅ All constraints satisfied!\n")
    formatted = format_complete_itinerary({
        "is_valid": True,
        "issues": [],
        "itinerary": validation["plan"],
        "final_message_to_user": selection.get("final_message_to_user", "Here is your plan!"),
    })
    print(formatted)
else:
    print("⚠️ Constraints not fully met:\n")
    negotiation_msg = format_negotiation_message(validation)
    print(negotiation_msg)
    print("\n--- To accept: run the next cell ---")

In [ ]:
# =============================================
# Step 4b: Accept closest option (run only if constraints were unmet)
# =============================================

plan_to_show = validation.get("closest_alternative") or validation["plan"]
formatted = format_complete_itinerary({
    "is_valid": False,
    "issues": validation.get("unmet_constraints", []),
    "itinerary": plan_to_show,
    "final_message_to_user": "Here is the closest feasible plan.",
})
print(formatted)